# 01 · Generate the Darcy-flow dataset (train / val / test)

Solves `-div(a ∇u) = 1` for random two-phase permeability fields `a(x)` with the **upstream
sampler and solver** (`src/data/darcy.py`, unchanged), in parallel across all CPU cores.
Each split uses its own seed, so val and test are truly held out.

**Kaggle settings:** Internet → On. No GPU needed (the solver is CPU-only).
Then **Save Version**: the `data/` folder becomes this notebook's output, which you attach as an
input to `02_train_ddp.ipynb`.

In [ ]:
import os

# ============================== CONFIG ==============================
REPO_URL   = "https://github.com/AbdullahRasheed45/fno-darcy-flow.git"  # upstream; src/ used as is
WORK_DIR   = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
GRID       = 64                                   # grid resolution (the app serves 64x64)
SPLITS     = {"train": (4000, 0),                 # name: (n_samples, seed)
              "val":   (500, 1),
              "test":  (500, 2)}
N_WORKERS  = os.cpu_count()                       # parallel solver processes
CHUNK_SIZE = 50                                   # samples per job; output is identical for any N_WORKERS
OUT_DIR    = f"{WORK_DIR}/data"
# ====================================================================

In [ ]:
# Setup: clone upstream (for src/), make it importable for this kernel and for subprocesses.
import subprocess, sys
REPO_DIR = f"{WORK_DIR}/fno-darcy-flow"
if not os.path.isdir(f"{REPO_DIR}/src"):
    subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(WORK_DIR)
os.environ["FNO_REPO"] = REPO_DIR
sys.path.insert(0, REPO_DIR)
print("repo:", REPO_DIR, "| cpus:", os.cpu_count(), "| workers:", N_WORKERS)

In [ ]:
%%writefile gen_darcy.py
"""Parallel Darcy dataset generation: train / val / test splits with separate seeds.

Written by notebooks/01_generate_dataset.ipynb. Reuses the upstream sampler and solver
(src/data/darcy.py) unchanged; only the work is split into chunks across processes.

    FNO_REPO=/path/to/fno-darcy-flow python gen_darcy.py \
        --splits train:4000:0,val:500:1,test:500:2 --grid 64 --workers 4 --out-dir data

Each chunk draws from its own np.random.default_rng([seed, chunk_id]), so the output is
identical for any --workers value. Different seeds per split keep the splits disjoint.
"""

from __future__ import annotations

import argparse
import json
import os
import sys
import time
from multiprocessing import Pool
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "1")  # one solver per core, no oversubscription
sys.path.insert(0, os.environ.get("FNO_REPO", os.getcwd()))

import numpy as np  # noqa: E402

from src.data.darcy import gaussian_random_field, permeability_from_grf, solve_darcy  # noqa: E402


def parse_splits(spec: str) -> dict[str, tuple[int, int]]:
    """'train:4000:0,val:500:1' -> {'train': (4000, 0), 'val': (500, 1)}"""
    out = {}
    for item in spec.split(","):
        name, n, seed = item.split(":")
        out[name] = (int(n), int(seed))
    return out


def solve_chunk(job: tuple[int, int, int, int]) -> tuple[np.ndarray, np.ndarray]:
    """job = (seed, chunk_id, n, grid) -> (a, u), each (n, grid, grid) float32."""
    seed, chunk_id, n, grid = job
    rng = np.random.default_rng([seed, chunk_id])
    a = np.empty((n, grid, grid), dtype=np.float32)
    u = np.empty((n, grid, grid), dtype=np.float32)
    for i in range(n):
        a[i] = permeability_from_grf(gaussian_random_field(grid, rng=rng))
        u[i] = solve_darcy(a[i])
    return a, u


def chunk_jobs(n: int, seed: int, grid: int, chunk_size: int) -> list[tuple[int, int, int, int]]:
    return [(seed, c, min(chunk_size, n - start), grid)
            for c, start in enumerate(range(0, n, chunk_size))]


def generate_split(n: int, seed: int, grid: int, workers: int,
                   chunk_size: int) -> tuple[np.ndarray, np.ndarray]:
    jobs = chunk_jobs(n, seed, grid, chunk_size)
    if workers <= 1:
        parts = [solve_chunk(j) for j in jobs]
    else:
        with Pool(workers) as pool:
            parts = pool.map(solve_chunk, jobs)
    return (np.concatenate([p[0] for p in parts]), np.concatenate([p[1] for p in parts]))


def main() -> None:
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument("--splits", default="train:4000:0,val:500:1,test:500:2")
    p.add_argument("--grid", type=int, default=64)
    p.add_argument("--workers", type=int, default=os.cpu_count() or 1)
    p.add_argument("--chunk-size", type=int, default=50)
    p.add_argument("--out-dir", type=Path, default=Path("data"))
    args = p.parse_args()

    args.out_dir.mkdir(parents=True, exist_ok=True)
    meta = {"grid": args.grid, "chunk_size": args.chunk_size, "workers": args.workers,
            "splits": {}}
    for name, (n, seed) in parse_splits(args.splits).items():
        t0 = time.time()
        a, u = generate_split(n, seed, args.grid, args.workers, args.chunk_size)
        dt = time.time() - t0
        out = args.out_dir / f"darcy_{name}.npz"
        np.savez_compressed(out, a=a, u=u)
        meta["splits"][name] = {"n": n, "seed": seed, "seconds": round(dt, 2), "file": out.name}
        print(f"{name}: {n} samples (seed {seed}) in {dt:.1f}s "
              f"({n / dt:.1f} samples/s, {args.workers} workers) -> {out}", flush=True)
    (args.out_dir / "meta.json").write_text(json.dumps(meta, indent=2))
    print(f"wrote {args.out_dir / 'meta.json'}")


if __name__ == "__main__":
    main()

In [ ]:
splits_arg = ",".join(f"{name}:{n}:{seed}" for name, (n, seed) in SPLITS.items())
!python gen_darcy.py --splits {splits_arg} --grid {GRID} --workers {N_WORKERS} --chunk-size {CHUNK_SIZE} --out-dir {OUT_DIR}

In [ ]:
# Sanity checks: shapes, finiteness, two-phase values, no sample shared between splits.
import numpy as np, json
data = {s: np.load(f"{OUT_DIR}/darcy_{s}.npz") for s in SPLITS}
for s, d in data.items():
    a, u = d["a"], d["u"]
    assert a.shape == u.shape == (SPLITS[s][0], GRID, GRID), (s, a.shape, u.shape)
    assert np.isfinite(a).all() and np.isfinite(u).all(), s
    assert set(np.unique(a)) <= {3.0, 12.0}, s
    print(f"{s:5s} a {a.shape}  frac(a=12) {np.mean(a == 12):.3f}   "
          f"u min {u.min():.2e} max {u.max():.2e} mean {u.mean():.2e}")
hashes = {s: {hash(x.tobytes()) for x in d["a"]} for s, d in data.items()}
names = list(hashes)
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        overlap = len(hashes[names[i]] & hashes[names[j]])
        assert overlap == 0, f"{names[i]}/{names[j]} share {overlap} samples"
print("splits are disjoint")
print(json.load(open(f"{OUT_DIR}/meta.json")))

In [ ]:
# A few samples per split: permeability a(x) and solution u(x).
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 6, figsize=(16, 5.5))
for k, s in enumerate(SPLITS):
    for j in range(2):
        col = 2 * k + j
        axes[0, col].imshow(data[s]["a"][j], cmap="viridis")
        axes[0, col].set_title(f"{s} #{j}  a(x)")
        im = axes[1, col].imshow(data[s]["u"][j], cmap="magma")
        axes[1, col].set_title("u(x)")
        fig.colorbar(im, ax=axes[1, col], fraction=0.046, pad=0.04)
for ax in axes.ravel():
    ax.set_xticks([]); ax.set_yticks([])
fig.tight_layout()
fig.savefig(f"{OUT_DIR}/samples.png", dpi=110)
plt.show()

**Next:** *Save Version* (Save & Run All). In `02_train_ddp.ipynb` use **Add Input → Your Work →
this notebook**; it auto-detects `darcy_train.npz` under `/kaggle/input`.